In [19]:
import pandas as pd
import numpy as np
import itertools
from typing import List, Dict, Any

# ==========================================================
# DATA PREPARATION
# ==========================================================
def prepare_sensor_data(
    df: pd.DataFrame,
    time_column: str = "time",
    sensor_list: List[str] = None
) -> pd.DataFrame:

    data = df.copy()
    data.columns = data.columns.str.strip()

    if sensor_list is None:
        sensor_list = [c for c in data.columns if c != time_column]

    sensor_list = [c.strip() for c in sensor_list]

    selected_cols = [time_column] + sensor_list
    data = data[selected_cols]

    data[time_column] = pd.to_datetime(
        data[time_column],
        unit="s",
        errors="coerce"
    )

    data = data.sort_values(time_column)
    data = data.set_index(time_column)

    num_cols = data.select_dtypes(include=np.number).columns

    if len(num_cols) > 0:
        data[num_cols] = data[num_cols].interpolate(
            method="linear",
            limit_direction="both"
        )

    data = data.dropna(how="all")

    return data


# ==========================================================
# CREATE WINDOWS
# ==========================================================
def generate_windows(
    df: pd.DataFrame,
    window_length: int = 100,
    shift: int = 10
) -> List[pd.DataFrame]:

    output = []
    total_rows = len(df)

    for i in range(0, total_rows - window_length + 1, shift):
        temp = df.iloc[i:i + window_length].copy()
        output.append(temp)

    return output


# ==========================================================
# CORRELATION PER WINDOW
# ==========================================================
def calculate_window_correlations(
    windows: List[pd.DataFrame],
    corr_type: str = "pearson"
) -> List[Dict[str, Any]]:

    final_result = []

    for n, win in enumerate(windows):

        if len(win) < 2:
            continue

        matrix = win.corr(method=corr_type)

        final_result.append({
            "window_no": n,
            "from_time": win.index[0],
            "to_time": win.index[-1],
            "rows": len(win),
            "matrix": matrix
        })

    return final_result


# ==========================================================
# DETECT CHANGES BETWEEN WINDOWS
# ==========================================================
def evaluate_changes(
    matrices: List[Dict[str, Any]]
) -> List[Dict[str, Any]]:

    variations = []

    if len(matrices) < 2:
        return variations

    for i in range(1, len(matrices)):

        old = matrices[i - 1]
        new = matrices[i]

        old_mat = old["matrix"]
        new_mat = new["matrix"]

        common_cols = sorted(
            list(set(old_mat.columns) & set(new_mat.columns))
        )

        for a, b in itertools.combinations(common_cols, 2):

            before = old_mat.loc[a, b]
            after = new_mat.loc[a, b]

            diff = abs(after - before)

            variations.append({
                "window_no": new["window_no"],
                "stream_a": a,
                "stream_b": b,
                "previous": float(before),
                "current": float(after),
                "change": float(diff),
                "start": new["from_time"],
                "end": new["to_time"]
            })

    return variations


# ==========================================================
# ALERT ENGINE
# ==========================================================
def build_alerts(
    change_data: List[Dict[str, Any]],
    strong_limit: float = 0.7,
    weak_limit: float = 0.4,
    min_change: float = 0.25
) -> List[Dict[str, Any]]:

    warnings = []

    for row in change_data:

        if row["change"] < min_change:
            continue

        prev = row["previous"]
        curr = row["current"]

        if prev >= strong_limit and curr <= weak_limit:
            severity = "HIGH"
            msg = "Strong relation dropped"

        elif (prev > 0 and curr < 0) or (prev < 0 and curr > 0):
            severity = "MEDIUM"
            msg = "Direction changed"

        else:
            severity = "LOW"
            msg = "Correlation changed"

        item = row.copy()
        item["severity"] = severity
        item["message"] = msg

        warnings.append(item)

    return warnings


# ==========================================================
# MAIN PIPELINE
# ==========================================================
def correlation_monitor(
    df: pd.DataFrame,
    time_column: str = "time",
    sensor_list: List[str] = None,
    window_length: int = 100,
    shift: int = 10
) -> Dict[str, Any]:

    cleaned = prepare_sensor_data(df, time_column, sensor_list)

    windows = generate_windows(
        cleaned,
        window_length,
        shift
    )

    matrices = calculate_window_correlations(windows)

    changes = evaluate_changes(matrices)

    alerts = build_alerts(changes)

    return {
        "cleaned_data": cleaned,
        "windows": windows,
        "matrices": matrices,
        "changes": changes,
        "alerts": alerts,
        "summary": {
            "windows_created": len(windows),
            "matrices_created": len(matrices),
            "changes_found": len(changes),
            "alerts_found": len(alerts)
        }
    }


# ==========================================================
# EXECUTION
# ==========================================================
if __name__ == "__main__":

    df = pd.read_csv("simple.csv")

    result = correlation_monitor(
        df=df,
        time_column="time",
        sensor_list=["s1", "s2", "s3"],
        window_length=100,
        shift=50
    )

    print("===== SUMMARY =====")
    print(result["summary"])
    print()

    print("===== WINDOW CORRELATION MATRICES =====\n")

    for item in result["matrices"]:

        print(
            f"Window {item['window_no']} | "
            f"{item['from_time']} --> {item['to_time']}"
        )

        print(item["matrix"].round(4))
        print("-" * 65)

    if result["alerts"]:
        print("\n===== ALERTS =====\n")

        alert_df = pd.DataFrame(result["alerts"])

        print(
            alert_df[
                [
                    "window_no",
                    "stream_a",
                    "stream_b",
                    "previous",
                    "current",
                    "change",
                    "severity"
                ]
            ].round(4)
        )

    else:
        print("\nNo alerts generated.")

===== SUMMARY =====
{'windows_created': 19, 'matrices_created': 19, 'changes_found': 54, 'alerts_found': 14}

===== WINDOW CORRELATION MATRICES =====

Window 0 | 1970-01-01 00:00:00 --> 1970-01-01 00:01:39
        s1      s2      s3
s1  1.0000 -0.9534  1.0000
s2 -0.9534  1.0000 -0.9534
s3  1.0000 -0.9534  1.0000
-----------------------------------------------------------------
Window 1 | 1970-01-01 00:00:50 --> 1970-01-01 00:02:29
        s1      s2      s3
s1  1.0000 -0.9605  1.0000
s2 -0.9605  1.0000 -0.9605
s3  1.0000 -0.9605  1.0000
-----------------------------------------------------------------
Window 2 | 1970-01-01 00:01:40 --> 1970-01-01 00:03:19
        s1      s2      s3
s1  1.0000 -0.4959  1.0000
s2 -0.4959  1.0000 -0.4959
s3  1.0000 -0.4959  1.0000
-----------------------------------------------------------------
Window 3 | 1970-01-01 00:02:30 --> 1970-01-01 00:04:09
        s1      s2      s3
s1  1.0000  0.9431  1.0000
s2  0.9431  1.0000  0.9431
s3  1.0000  0.9431  1.0000